# Movie Review Sentiment Analysis

Walkthrough of the whole pipeline: **data → TF-IDF → logistic regression → evaluation → predictions**.
The same code lives in `train.py` (training) and `sentiment.py` (prediction); this notebook explains it step by step.

> **Honest note on data.** `data/reviews.csv` is *template-generated* (see `generate_sample_data.py`), so it is easy to separate
> and scores ~100%. The realistic number is the accuracy on the **held-out hand-written reviews** at the end.
> To get a real-world number, train on a real dataset such as IMDB (50k reviews): `python train.py --data your_imdb.csv`.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import train, sentiment

main_df, hand = train.load_data("data/reviews.csv")
print(main_df.shape, hand.shape)
main_df.sample(5, random_state=1)

## 1. Explore the data

In [ ]:
main_df["words"] = main_df.review.str.split().str.len()
print(main_df.sentiment.value_counts().rename({1: "positive", 0: "negative"}))
print(main_df.words.describe().round(1))
main_df.groupby("sentiment").words.plot.hist(alpha=.6, bins=15, legend=True)
plt.xlabel("words per review"); plt.show()

## 2. Features: TF-IDF
Each review becomes a vector. Words that are frequent in one review but rare overall get high weights.
Stop-words are removed, **except negations** ("not", "no", "never"), because "not good" means the opposite of "good".
Bigrams ("not good", "waste time") are included for the same reason.

In [ ]:
pipe = train.make_pipeline()
tfidf = pipe.named_steps["tfidf"].fit(main_df.review)
print(len(tfidf.get_feature_names_out()), "features")
print(tfidf.get_feature_names_out()[:15])

## 3. Train / test split and baseline model

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(main_df.review, main_df.sentiment, test_size=.2,
                                          random_state=42, stratify=main_df.sentiment)
base = train.make_pipeline().fit(X_tr, y_tr)
ht = hand[hand.split == "test"]
print("template test accuracy   :", (base.predict(X_te) == y_te).mean().round(3))
print("hand-written test accuracy:", (base.predict(ht.review) == ht.sentiment).mean().round(3), f"(n={len(ht)})")

The baseline is perfect on template data and noticeably worse on natural text: it only learned the template vocabulary.
`train.py` therefore also trains on a small set of **hand-written** reviews (up-weighted) and tunes `C` and that weight with
cross-validation on those reviews only. The hand-written *test* reviews are never used for training or tuning.

## 4. Run the full training script and read the results

In [ ]:
import json, subprocess, sys
subprocess.run([sys.executable, "train.py"], check=True, capture_output=True)
res = json.load(open("model/metrics.json"))
pd.DataFrame({"template test": res["final_model"]["on_main_test"],
              "hand-written test": res["final_model"]["on_handwritten_test"]})

## 5. Evaluation on the hand-written held-out reviews

In [ ]:
model = sentiment.get_model()
pred = model.predict(ht.review)
print(classification_report(ht.sentiment, pred, target_names=["Negative", "Positive"]))
cm = confusion_matrix(ht.sentiment, pred)
plt.imshow(cm, cmap="Blues"); plt.xticks([0, 1], ["Neg", "Pos"]); plt.yticks([0, 1], ["Neg", "Pos"])
for i in range(2):
    for j in range(2): plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=16)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.show()

### Mistakes (error analysis)

In [ ]:
wrong = ht[pred != ht.sentiment]
for r, p in zip(wrong.review, pred[pred != ht.sentiment]):
    print(f"predicted {'pos' if p else 'neg'}: {r}")

## 6. What did the model learn?

In [ ]:
print(res["top_terms"])
from IPython.display import Image
Image("visualizations/top_words.png")

## 7. Predict your own review

In [ ]:
for text in ["A beautiful, moving film. Loved every minute.",
             "This is not a good movie.",
             "Boring, predictable and way too long."]:
    r = sentiment.predict(text)
    print(f"{r['label']:9s} {r['confidence']:.0%}  {text}   <- {[w['word'] for w in r['top_words'][:3]]}")

## Limits and next steps
* The hand-written test set is only 30 reviews, so its accuracy has a wide error bar (roughly ±12 points).
* Train on IMDB (50k real reviews) for a trustworthy number; expect roughly 88-90% with this same pipeline.
* Sarcasm, mixed reviews and long context are out of reach for a bag-of-words model; try a transformer (e.g. DistilBERT) next.